# Does Weather Affect the Stock Market? A BIST 100 Analysis
**DSA210 – Phase 2: Hypothesis Testing**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load the pre-processed merged dataset saved in Phase 1
df = pd.read_csv("../data/merged.csv", index_col="date", parse_dates=True)
print(f"Dataset loaded: {df.shape}")
df.head()

Dataset loaded: (496, 7)


,open,close,volume,daily_return,temp_mean,precipitation,windspeed
date,,,,,,,
2023-01-02,5568.370125,5661.069824,4834734400,1.664755,6.8,0.0,10.6
2023-01-03,5697.569937,5626.570312,5459192700,-1.246139,6.9,0.0,17.6
2023-01-04,5646.570148,5523.470703,4992602800,-2.180075,8.6,0.1,15.9
2023-01-05,5555.970307,5116.372559,4510533000,-7.912169,9.4,0.0,16.6
2023-01-06,5120.972851,5341.971680,5331359700,4.315563,9.4,4.0,19.5


## 5. Hypothesis Testing

We test whether weather variables have a statistically significant relationship with BIST 100 daily returns.

- **H0 (null hypothesis):** The weather variable has no significant relationship with daily returns.
- **H1 (alternative hypothesis):** The weather variable has a significant relationship with daily returns.
- **Significance level:** α = 0.05

In [2]:
from scipy import stats

# --- Pearson and Spearman Correlation Tests ---
# Pearson: assumes linear relationship and normal distribution
# Spearman: non-parametric, tests monotonic relationship without normality assumption

correlation_results = []
weather_vars = ["temp_mean", "precipitation", "windspeed"]
labels = ["Temperature", "Precipitation", "Windspeed"]

for var, label in zip(weather_vars, labels):
    # Drop any missing values before testing
    common = df[[var, "daily_return"]].dropna()
    x, y = common[var], common["daily_return"]

    pearson_r, pearson_p = stats.pearsonr(x, y)
    spearman_r, spearman_p = stats.spearmanr(x, y)

    correlation_results.append({
        "Variable": label,
        "Pearson r": round(pearson_r, 4),
        "Pearson p-value": round(pearson_p, 4),
        "Spearman r": round(spearman_r, 4),
        "Spearman p-value": round(spearman_p, 4),
        "Significant (alpha=0.05)": "Yes" if min(pearson_p, spearman_p) < 0.05 else "No"
    })

corr_results_df = pd.DataFrame(correlation_results)
print("=== Pearson & Spearman Correlation Tests ===\n")
corr_results_df

=== Pearson & Spearman Correlation Tests ===



,Variable,Pearson r,Pearson p-value,Spearman r,Spearman p-value,Significant (alpha=0.05)
0,Temperature,0.0231,0.6080,0.0177,0.6941,No
1,Precipitation,0.0008,0.9853,0.0800,0.0750,No
2,Windspeed,0.0480,0.2857,0.0705,0.1169,No


In [3]:
# --- Mann-Whitney U Test: Rainy vs Non-Rainy Days ---
# Tests whether market returns on rainy days differ significantly from dry days.
# Mann-Whitney is used instead of t-test because precipitation is heavily skewed.

rainy = df[df["precipitation"] > 0]["daily_return"]
dry   = df[df["precipitation"] == 0]["daily_return"]

stat, p_val = stats.mannwhitneyu(rainy, dry, alternative="two-sided")

print("=== Mann-Whitney U Test: Rainy vs Non-Rainy Days ===\n")
print(f"  Rainy days   : n={len(rainy):3d} | mean return = {rainy.mean():.4f}%")
print(f"  Non-rainy days: n={len(dry):3d} | mean return = {dry.mean():.4f}%")
print(f"  U-statistic  : {stat:.2f}")
print(f"  p-value      : {p_val:.4f}")
print(f"\n  Conclusion   : {'Reject H0 — returns differ significantly on rainy vs dry days' if p_val < 0.05 else 'Fail to reject H0 — no significant difference between rainy and dry days'}")

=== Mann-Whitney U Test: Rainy vs Non-Rainy Days ===

  Rainy days   : n=251 | mean return = -0.0335%
  Non-rainy days: n=245 | mean return = -0.2681%
  U-statistic  : 33241.00
  p-value      : 0.1183

  Conclusion   : Fail to reject H0 — no significant difference between rainy and dry days


In [4]:
# --- Independent t-test: Hot vs Cold Days ---
# Splits the dataset by median temperature and tests whether
# returns on hot days are statistically different from cold days.

median_temp = df["temp_mean"].median()
hot  = df[df["temp_mean"] >= median_temp]["daily_return"]
cold = df[df["temp_mean"] <  median_temp]["daily_return"]

stat, p_val = stats.ttest_ind(hot, cold)

print("=== Independent t-test: Hot vs Cold Days ===\n")
print(f"  Median temperature : {median_temp:.1f} C")
print(f"  Hot days  (>= median): n={len(hot):3d} | mean return = {hot.mean():.4f}%")
print(f"  Cold days (<  median): n={len(cold):3d} | mean return = {cold.mean():.4f}%")
print(f"  t-statistic: {stat:.4f}")
print(f"  p-value    : {p_val:.4f}")
print(f"\n  Conclusion : {'Reject H0 — significant difference between hot and cold day returns' if p_val < 0.05 else 'Fail to reject H0 — no significant difference between hot and cold day returns'}")

=== Independent t-test: Hot vs Cold Days ===

  Median temperature : 14.9 C
  Hot days  (>= median): n=248 | mean return = -0.1503%
  Cold days (<  median): n=248 | mean return = -0.1485%
  t-statistic: -0.0112
  p-value    : 0.9911

  Conclusion : Fail to reject H0 — no significant difference between hot and cold day returns


### Hypothesis Testing Summary

| Test | Variables | Result |
|------|-----------|--------|
| Pearson / Spearman | Temperature vs Return | See table above |
| Pearson / Spearman | Precipitation vs Return | See table above |
| Pearson / Spearman | Windspeed vs Return | See table above |
| Mann-Whitney U | Rainy vs Non-Rainy Days | See output above |
| Independent t-test | Hot vs Cold Days | See output above |

**Overall Conclusion:** The hypothesis tests consistently show p-values above the 0.05 threshold, indicating that weather variables alone do not have a statistically significant effect on BIST 100 daily returns. This aligns with the efficient market hypothesis, which suggests that publicly available information (including weather) is already priced in.

### Visualizations: Group Comparisons

In [ ]:
from scipy.stats import norm

rainy = df[df["precipitation"] > 0]["daily_return"]
dry   = df[df["precipitation"] == 0]["daily_return"]
median_temp = df["temp_mean"].median()
hot  = df[df["temp_mean"] >= median_temp]["daily_return"]
cold = df[df["temp_mean"] <  median_temp]["daily_return"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Rainy vs Dry: KDE + box ---
for data, label, color in [(dry, "Dry", "#AED6F1"), (rainy, "Rainy", "#85C1E9")]:
    axes[0].hist(data, bins=30, density=True, alpha=0.45, color=color, edgecolor="white", label=label)
    xr = np.linspace(data.min(), data.max(), 200)
    axes[0].plot(xr, norm.pdf(xr, data.mean(), data.std()), color=color, linewidth=2)
    axes[0].axvline(data.mean(), color=color, linestyle="--", linewidth=1.5,
                    label=f"{label} mean: {data.mean():.3f}%")
axes[0].set_title("Return Distribution: Rainy vs Dry Days")
axes[0].set_xlabel("Daily Return (%)")
axes[0].set_ylabel("Density")
axes[0].legend()

# --- Hot vs Cold: KDE + box ---
for data, label, color in [(cold, "Cold", "#A9DFBF"), (hot, "Hot", "#F0B27A")]:
    axes[1].hist(data, bins=30, density=True, alpha=0.45, color=color, edgecolor="white", label=label)
    xr = np.linspace(data.min(), data.max(), 200)
    axes[1].plot(xr, norm.pdf(xr, data.mean(), data.std()), color=color, linewidth=2)
    axes[1].axvline(data.mean(), color=color, linestyle="--", linewidth=1.5,
                    label=f"{label} mean: {data.mean():.3f}%")
axes[1].set_title("Return Distribution: Hot vs Cold Days")
axes[1].set_xlabel("Daily Return (%)")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.suptitle("Group Comparisons — BIST 100 Daily Returns", fontsize=13)
plt.tight_layout()
plt.show()